# RQ2 Assumption Checks: Ranking Stability (Kendall's τ)

This notebook checks the basic assumptions behind the Kendall's τ
statistics used in `experiment/run_rq2_experiment.py`.

For RQ2, the core data are:

- **Per perturbation**: one Kendall's τ comparing the baseline ranking R₀
  to the ranking after a one-note change (add/remove favourite/avoid).
- **Per baseline**: the mean τ across all perturbations for that baseline
  profile.

The experiment treats baselines as the unit of analysis when computing the
95% confidence interval (bootstrap over baseline means).

Here we:

- load the actual per-perturbation and per-baseline τ values from
  `experiment_results/RQ2_results.json`,
- visualise the distribution of τ across perturbations and baselines,
- and discuss what these plots tell us about the assumptions behind our
  RQ2 metrics.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# This notebook lives in experiment_results/assumption_checks
EXP_DIR = Path("..")
RQ2_PATH = EXP_DIR / "RQ2_results.json"

with RQ2_PATH.open(encoding="utf-8") as f:
    rq2 = json.load(f)

per_pert = rq2["per_perturbation"]
baseline_profiles = rq2["baseline_profiles"]

taus = [p["tau"] for p in per_pert]
pert_types = [p["perturbation_type"] for p in per_pert]
baseline_tau = [b["mean_tau"] for b in baseline_profiles]

len(taus), min(taus), max(taus)


In [ ]:
# Histogram of all Kendall tau values (per perturbation)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(taus, bins=10, edgecolor="black", alpha=0.8)
ax.set_xlabel("Kendall's tau (per perturbation)")
ax.set_ylabel("Number of perturbations")
ax.set_title("RQ2: Distribution of tau across all perturbations")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


The histogram above shows how similar the new rankings are to the baseline
ranking, across **all** one-note perturbations.

- Values close to 1 indicate almost no change in order.
- Values near 0 would indicate weak relationship.
- Negative values would indicate reversals (strong disagreements).

For Kendall's τ, we mainly need to confirm that:

- τ values stay within [−1, 1],
- there are no NaNs (the experiment code already raises on NaN),
- the distribution is sensible and not dominated by obvious artefacts.


In [ ]:
# Histogram of baseline mean tau values (unit of analysis for CI)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(baseline_tau, bins=5, edgecolor="black", alpha=0.8)
ax.set_xlabel("Mean Kendall's tau per baseline")
ax.set_ylabel("Number of baselines")
ax.set_title("RQ2: Distribution of baseline-level mean tau")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()


The baseline-level means above are the values over which the bootstrap
confidence interval is computed (resampling baselines with replacement).

Things to look for:

- Are baseline means all in a plausible range (e.g. between about 0.7
  and 0.9 as reported in the main results)?
- Is there any extreme outlier baseline that would dominate the mean?

As long as there are multiple baselines with reasonable mean τ values,
bootstrapping over these means is an appropriate way to quantify
uncertainty in "average ranking stability".
